# Unit 6: 高级专题与系统集成

## 学习目标

- 掌握 Daisy 16 通道扩展模块的配置与使用
- 理解 Micro SD 卡离线数据记录的格式和读取方法
- 了解 Cyton 固件更新流程和自定义编程基础
- 学习 BrainFlow 采集数据存储为标准格式（EDF/BDF）
- 构建完整的端到端 BCI 数据处理管线

---

## 6.1 Daisy 16 通道扩展

Daisy 模块通过 GPIO 连接到 Cyton 主板，将通道数扩展至 16。

### 硬件连接

```
┌───────────┐    GPIO 排线    ┌───────────┐
│   Cyton   │◄──────────────►│   Daisy   │
│ CH1-CH8   │                │ CH9-CH16  │
└───────────┘                └───────────┘
```

### BrainFlow 中的 Daisy 配置

- **Board ID**: 2 (BoardIds.CYTON_DAISY_BOARD)
- **采样率**: 125 SPS (16 通道时降低为一半)
- **数据包**: 33 字节 × 2（交替传输 CH1-8 和 CH9-16）

### SDK 命令

| 命令 | 功能 | 说明 |
|------|------|------|
| `C` | 切换 Daisy 模式 | `C` 切换 Daisy 扩展 |
| `c` | 查询 Daisy 状态 | 返回 Daisy 是否激活 |

### Daisy 10-20 通道映射

| 物理通道 | 10-20 | 推荐用途 |
|----------|-------|----------|
| CH9 | Fz | 中线额叶 |
| CH10 | Cz | 中线中央 (常用参考) |
| CH11 | Pz | 中线顶叶 (P300) |
| CH12 | Oz | 中线枕叶 (SSVEP) |
| CH13 | F3 | 左额叶 |
| CH14 | F4 | 右额叶 |
| CH15 | T7 | 左颞叶 |
| CH16 | T8 | 右颞叶 |

In [ ]:
# ============================================================
# 代码 6.1: Daisy 16 通道 BrainFlow 数据采集演示
# ============================================================
"""
Daisy 16 通道数据采集。需要真实硬件或使用 Synthetic Board 模拟。
"""
from utils import create_cyton_board, CYTON_8CH_1020_MAP, DAISY_8CH_1020_MAP
import time
import numpy as np
import matplotlib.pyplot as plt

SFREQ_16CH = 125.0  # Daisy 模式下采样率

# 完整的 16 通道映射
FULL_16CH_MAP = {**CYTON_8CH_1020_MAP, **DAISY_8CH_1020_MAP}

print("16-Channel Cyton + Daisy Configuration")
print("=" * 50)
print(f"{'Physical CH':<12} {'10-20 Label':<8} {'Recommended Use':<25}")
print("-" * 50)

uses = {
    1: "Frontal EEG", 2: "Frontal EEG", 3: "Motor Imagery (C3)",
    4: "Motor Imagery (C4)", 5: "Visual ERPs", 6: "Visual ERPs",
    7: "Alpha/VEP (O1)", 8: "Alpha/VEP (O2)",
    9: "Midline Frontal", 10: "Reference/Midline", 11: "P300 (Pz)",
    12: "SSVEP (Oz)", 13: "Left Frontal", 14: "Right Frontal",
    15: "Left Temporal", 16: "Right Temporal"
}

for ch in range(1, 17):
    label = FULL_16CH_MAP.get(ch, "—")
    use = uses.get(ch, "—")
    print(f"CH{ch:<11} {label:<8} {use:<25}")

print("\nDaisy Hardware Setup:")
print("  1. Power off Cyton (remove battery)")
print("  2. Connect Daisy module to Cyton GPIO pins (align J1 connector)")
print("  3. Power on (Daisy LED should light up)")
print("  4. Use Board ID = 2 in BrainFlow")

In [ ]:
# ============================================================
# 代码 6.2: 16 通道模拟信号生成与可视化
# ============================================================
from utils import generate_synthetic_eeg
import numpy as np
import matplotlib.pyplot as plt

# 生成 16 通道模拟数据
data_16ch, times = generate_synthetic_eeg(
    duration=5.0, sfreq=SFREQ_16CH, n_channels=16,
    noise_level=5.0, alpha_power=25.0, seed=500
)

# 为额外通道添加区域特异性特征
t = times
rng = np.random.default_rng(501)
for ch in range(8, 16):
    # 中线通道 (Fz, Cz, Pz, Oz) 添加中线特征
    if ch in [8, 9, 10, 11]:
        data_16ch[ch] += 15 * np.sin(2 * np.pi * 5 * t)  # theta
    # 颞叶通道 (T7, T8) 添加 gamma
    if ch in [14, 15]:
        data_16ch[ch] += 8 * np.sin(2 * np.pi * 35 * t)

print(f"16-channel simulated data shape: {data_16ch.shape}")
print(f"Duration: {times[-1]:.1f}s @ {SFREQ_16CH} Hz")

# 快速概览：前 2 秒的所有 16 通道
fig, axes = plt.subplots(16, 1, figsize=(14, 14), sharex=True)

mask = times <= 2.0
for ch in range(16):
    label = FULL_16CH_MAP.get(ch+1, f"CH{ch+1}")
    axes[ch].plot(times[mask], data_16ch[ch][mask], linewidth=0.3,
                  color="steelblue" if ch < 8 else "coral")
    axes[ch].set_ylabel(f"{label}", fontsize=7, rotation=0, labelpad=20)
    axes[ch].set_ylim(-80, 80)
    axes[ch].tick_params(labelsize=6)

axes[0].set_title("16-Channel EEG: Cyton (CH1-8, blue) + Daisy (CH9-16, coral)",
                  fontsize=12, fontweight="bold")
axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

## 6.2 Micro SD 卡离线记录

Cyton 支持将数据直接写入 Micro SD 卡，实现完全无线的离线记录。

### SD 卡命令集（通过串口发送）

| 命令 | 功能 | 说明 |
|------|------|------|
| `A` | 初始化 SD 卡 | 上电后必须先执行 |
| `S` | 创建新文件 | 自动生成文件名 (如 `BCI_001.txt`) |
| `F` | 关闭当前文件 | 安全关闭，确保数据写入 |
| `G` | 开始记录 | 数据开始写入 SD 卡 |
| `H` | 停止记录 | 暂停写入（不关闭文件） |
| `J` | 追加记录 | 在已存在文件后追加数据 |

### SD 卡数据格式

```
%OpenBCI Raw EEG Data
%Sample Rate = 250 SPS
%Number of channels = 8
%Board Mode = DEFAULT
SampleIndex, CH1, CH2, CH3, CH4, CH5, CH6, CH7, CH8, AccelX, AccelY, AccelZ, Timestamp
1, -1234.5, 567.8, -2345.6, ... , 0.004
2, -1235.0, 568.2, -2346.1, ... , 0.008
```

In [ ]:
# ============================================================
# 代码 6.3: SD 卡离线数据读取与解析
# ============================================================
import numpy as np
import pandas as pd
import os


def generate_sample_sd_file(filepath, duration_sec=10.0, sfreq=250.0):
    """
    生成一个模拟的 OpenBCI SD 卡数据文件（用于测试解析代码）。
    
    真实 SD 卡数据文件可通过 OpenBCI GUI 或 SDK 命令获得。
    """
    n_samples = int(duration_sec * sfreq)
    t = np.arange(n_samples) / sfreq
    
    with open(filepath, "w") as f:
        # 文件头
        f.write("%OpenBCI Raw EEG Data\n")
        f.write(f"%Sample Rate = {sfreq} SPS\n")
        f.write("%Number of channels = 8\n")
        f.write("%Board Mode = DEFAULT\n")
        # 列名
        f.write("SampleIndex, CH1, CH2, CH3, CH4, CH5, CH6, CH7, CH8, "
                "AccelX, AccelY, AccelZ, Timestamp\n")
        
        rng = np.random.default_rng(42)
        for i in range(n_samples):
            ch_data = rng.normal(0, 10, 8)  # 8 通道数据 (muV)
            ch_data[6] += 30 * np.sin(2 * np.pi * 10 * t[i])  # O1 alpha
            ch_data[7] += 30 * np.sin(2 * np.pi * 10 * t[i])  # O2 alpha
            
            accel = rng.normal(0, 50, 3)
            ts = i / sfreq
            
            line = f"{i+1}," + ",".join([f"{v:.2f}" for v in ch_data]) + \
                   "," + ",".join([f"{v:.1f}" for v in accel]) + \
                   f",{ts:.3f}\n"
            f.write(line)
    
    print(f"Generated sample SD file: {filepath}")
    print(f"  Duration: {duration_sec}s, Samples: {n_samples}")
    return filepath


def parse_sd_file(filepath):
    """
    解析 OpenBCI SD 卡数据文件。
    
    Returns
    -------
    df : DataFrame
        解析后的数据表
    metadata : dict
        文件头元数据
    """
    metadata = {}
    
    with open(filepath, "r") as f:
        lines = f.readlines()
    
    # 解析头部元数据
    header_end = 0
    for i, line in enumerate(lines):
        if line.startswith("%"):
            if "Sample Rate" in line:
                metadata["sample_rate"] = float(line.split("=")[1].strip().split()[0])
            elif "Number of channels" in line:
                metadata["n_channels"] = int(line.split("=")[1].strip())
            elif "Board Mode" in line:
                metadata["board_mode"] = line.split("=")[1].strip()
        elif line.startswith("SampleIndex"):
            header_end = i
            break
    
    # 读取数据
    from io import StringIO
    data_csv = "".join(lines[header_end:])
    df = pd.read_csv(StringIO(data_csv))
    
    return df, metadata


# ---- 生成并解析一个示例文件 ----
import tempfile
sample_sd_path = "../data/sample_sd_data.txt"
os.makedirs("../data", exist_ok=True)

generate_sample_sd_file(sample_sd_path, duration_sec=5.0, sfreq=250.0)

df, meta = parse_sd_file(sample_sd_path)
print(f"\nParsed SD file: {sample_sd_path}")
print(f"Metadata: {meta}")
print(f"Data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())

# 快速可视化
fig, ax = plt.subplots(figsize=(12, 4))
for ch in ["CH7", "CH8"]:
    ax.plot(df["Timestamp"], df[ch], linewidth=0.5, label=ch)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (muV)")
ax.set_title("SD Card Recorded Data: Occipital Channels (Alpha)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6.3 EDF/BDF 标准格式导出

EDF (European Data Format) 和 BDF (BioSemi Data Format) 是 EEG 数据的国际标准格式，
广泛支持于 EEGLAB、FieldTrip、MNE 等分析工具。

### 格式对比

| 特性 | EDF | EDF+ | BDF |
|------|-----|------|-----|
| 位深度 | 16-bit | 16-bit | 24-bit |
| 文件大小限制 | 61440 samples | 无限制 | 无限制 |
| 注释支持 | 无 | 有 | 有 |
| Cyton 适用 | ✓ | ✓ | ✓ (推荐) |

In [ ]:
# ============================================================
# 代码 6.4: 使用 MNE 导出 EDF 格式文件
# ============================================================
"""
将 OpenBCI 数据导出为 EDF/BDF 格式以便与其他工具共享。
"""
import numpy as np
from utils import generate_synthetic_eeg, CYTON_8CH_1020_MAP

# 生成数据
raw_data, _ = generate_synthetic_eeg(duration=10.0, sfreq=250.0, n_channels=8, seed=99)
channel_names = [CYTON_8CH_1020_MAP.get(i+1, f"CH{i+1}") for i in range(8)]

print("Using MNE to export EDF...")
try:
    import mne
    
    # 创建 MNE info 对象
    ch_types = ["eeg"] * 8
    info = mne.create_info(ch_names=channel_names, sfreq=250.0, ch_types=ch_types)
    
    # 创建 Raw 对象
    raw = mne.io.RawArray(raw_data, info)
    
    # 导出 EDF
    edf_path = "../data/openbci_cyton_export.edf"
    mne.export.export_raw(edf_path, raw, fmt="edf", overwrite=True)
    print(f"Exported to: {edf_path}")
    
    # 验证：重新读取
    raw_loaded = mne.io.read_raw_edf(edf_path, preload=True)
    print(f"Verified: {raw_loaded.ch_names}")
    print(f"  Duration: {raw_loaded.times[-1]:.1f}s")
    print(f"  Data shape: {raw_loaded.get_data().shape}")
    
except ImportError:
    print("MNE not installed. Install with: pip install mne")
    print("\nManual EDF header construction (demonstration only):")
    print("""
    # EDF header (ASCII):
    # - 256 bytes: version, patient ID, recording ID
    # - Start date, start time
    # - Number of bytes in header record
    # - Reserved
    # - Number of data records
    # - Duration of each data record
    # - Number of signals
    # - For each signal: label, transducer, dimension, physical min/max,
    #   digital min/max, prefiltering, samples per record, reserved
    """)

## 6.4 Cyton 固件编程简介

OpenBCI Cyton 的固件基于 Arduino/MPIDE 框架，使用 PIC32 编译器。

### 固件架构（v3.0.0+）

```
OpenBCI_32bit_Library/
├── OpenBCI_32bit_Library.ino    # 主程序
├── ADS1299.cpp/h                # ADS1299 驱动
├── LIS3DH.cpp/h                 # 加速度计驱动
├── OBCI_SD.cpp/h                # SD 卡驱动
├── Radio.cpp/h                  # RFDuino 通信
└── ...
```

### 编程工具链

1. **Arduino IDE** 或 **MPIDE**（PIC32 专用）
2. **chipKIT-core** 开发板包
3. **PIC32 编译器**（XC32）
4. **USB Dongle 作为编程器**（通过 FTDI 芯片）

### 固件上传步骤

1. 将 Cyton 通过 USB Dongle 连接
2. 切换 Cyton 到编程模式（按住 PROG 按钮上电）
3. 在 Arduino IDE 中选择 "chipKIT UDB32-MX2-DIP" 开发板
4. 选择正确的串口
5. 编译并上传

### 关键代码示例（修改采样率）

```cpp
// 在 OpenBCI_32bit_Library.ino 中
// 修改 ADS1299 采样率

// CONFIG1 register (address 0x01)
// Bits [2:0] = DR[2:0]:
//   110 = 250 SPS (default)
//   101 = 500 SPS
//   100 = 1000 SPS

ads.sdata[ADS1299::CONFIG1] = 0b10010110;  // 250 SPS
// ads.sdata[ADS1299::CONFIG1] = 0b10010101;  // 500 SPS
```

In [ ]:
# ============================================================
# 代码 6.5: 固件命令测试工具 (Python)
# ============================================================
"""
通过串口直接发送 SDK 命令到 Cyton 设备。
用于测试和调试固件功能。
"""

CYTON_SDK_COMMANDS = {
    # 流控制
    "start_stream":   {"cmd": b"b", "desc": "Toggle binary stream on/off"},
    "stop_stream":    {"cmd": b"s", "desc": "Stop data stream"},
    "soft_reset":     {"cmd": b"v", "desc": "Software reset"},
    
    # 通道配置
    "channel_default": {"cmd": b"d", "desc": "Set all channels to default"},
    "channel_on_1-8":  {"cmd": b"x11111111 10000X", "desc": "Enable all 8 channels + accel"},
    
    # 增益设置
    "gain_ch1_x24":  {"cmd": b"110X", "desc": "CH1 gain = 24"},
    "gain_ch1_x12":  {"cmd": b"101X", "desc": "CH1 gain = 12"},
    
    # 测试信号
    "test_signal_dc":    {"cmd": b"001000X", "desc": "DC test signal"},
    "test_signal_square": {"cmd": b"001001X", "desc": "1Hz square wave test"},
    "test_signal_off":    {"cmd": b"000000X", "desc": "Test signal off"},
    
    # 阻抗测试
    "impedance_ch1":       {"cmd": b"z1X", "desc": "Impedance test CH1"},
    "impedance_all":       {"cmd": b"ZX", "desc": "Impedance test all channels"},
    
    # SD 卡
    "sd_init":    {"cmd": b"A", "desc": "Initialize SD card"},
    "sd_newfile":  {"cmd": b"S", "desc": "Create new SD file"},
    "sd_start":   {"cmd": b"G", "desc": "Start SD recording"},
    "sd_stop":    {"cmd": b"H", "desc": "Stop SD recording"},
    "sd_close":   {"cmd": b"F", "desc": "Close SD file"},
    
    # Daisy
    "daisy_toggle": {"cmd": b"C", "desc": "Toggle Daisy mode"},
    "daisy_query":  {"cmd": b"c", "desc": "Query Daisy status"},
}

print("OpenBCI Cyton SDK Commands Reference")
print("=" * 70)
for category in ["Stream Control", "Channel Config", "Gain", "Test Signal",
                  "Impedance", "SD Card", "Daisy"]:
    print(f"\n--- {category} ---")
    for name, info in CYTON_SDK_COMMANDS.items():
        cmd_str = info["cmd"].decode("ascii", errors="replace").strip("X")
        print(f"  {cmd_str:<25} {info['desc']}")

print("\n\nNote on command termination:")
print("- Commands ending with 'X' require '\n' (0x0A) termination")
print("- Commands without 'X' are immediate (no termination needed)")

## 6.5 端到端 BCI 数据处理管线

整合前面所有单元的知识，构建完整的管线：

In [ ]:
# ============================================================
# 代码 6.6: 端到端 BCI 数据处理管线
# ============================================================
"""
完整的端到端 BCI 数据处理管线，整合预处理、特征提取、分类。
"""
import numpy as np
from scipy import signal
from scipy.linalg import eigh
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from utils import generate_synthetic_eeg, compute_band_powers


class EndToEndBCIPipeline:
    """
    端到端 BCI 数据处理管线。
    
    整合了:
    1. 带通滤波 (8-30 Hz for MI)
    2. CSP 空间滤波
    3. 特征提取 (log-variance)
    4. LDA 分类
    5. 频带功率分析（用于解释）
    """
    
    def __init__(self, sfreq=250.0, n_csp_components=4):
        self.sfreq = sfreq
        self.n_csp_components = n_csp_components
        self.b_bp, self.a_bp = signal.butter(
            4, [8/(sfreq/2), 30/(sfreq/2)], btype="band")
        self.csp_filters = None
        self.classifier = None
        self.is_fitted = False
    
    def preprocess(self, data):
        """带通滤波"""
        if data.ndim == 2:
            return signal.filtfilt(self.b_bp, self.a_bp, data, axis=-1)
        elif data.ndim == 3:
            return np.array([signal.filtfilt(self.b_bp, self.a_bp, d, axis=-1)
                             for d in data])
    
    def extract_csp_features(self, data):
        """CSP 特征提取"""
        features = []
        for trial in data:
            filtered = self.csp_filters @ trial
            feat = np.log(np.var(filtered, axis=1) + 1e-10)
            features.append(feat)
        return np.array(features)
    
    def fit(self, X, y):
        """
        训练管线。
        X: (n_trials, n_channels, n_samples)
        y: (n_trials,) labels
        """
        # 预处理
        X_clean = self.preprocess(X)
        
        # CSP 训练
        n_channels = X.shape[1]
        X1 = X_clean[y == 0]
        X2 = X_clean[y == 1]
        cov1 = np.mean([np.cov(t) for t in X1], axis=0) + 1e-6 * np.eye(n_channels)
        cov2 = np.mean([np.cov(t) for t in X2], axis=0) + 1e-6 * np.eye(n_channels)
        eigvals, eigvecs = eigh(cov1, cov1 + cov2)
        idx = np.argsort(eigvals)[::-1]
        eigvecs = eigvecs[:, idx]
        half = self.n_csp_components // 2
        self.csp_filters = np.hstack([eigvecs[:, :half],
                                      eigvecs[:, -half:]]).T
        
        # 提取特征
        features = self.extract_csp_features(X_clean)
        
        # 训练分类器
        self.classifier = LinearDiscriminantAnalysis()
        self.classifier.fit(features, y)
        
        self.is_fitted = True
        return self
    
    def predict(self, X):
        """预测单个/多个 trial"""
        if not self.is_fitted:
            raise RuntimeError("Pipeline not fitted. Call .fit() first.")
        
        X_clean = self.preprocess(X)
        if X_clean.ndim == 2:
            X_clean = X_clean[np.newaxis, :, :]
        features = self.extract_csp_features(X_clean)
        pred = self.classifier.predict(features)
        proba = self.classifier.predict_proba(features)
        return pred, proba


# ---- 演示 ----
print("=" * 60)
print("End-to-End BCI Pipeline Demonstration")
print("=" * 60)

# 生成数据
from unit_05_bci_applications import generate_mi_dataset
X, y = generate_mi_dataset(n_trials_per_class=50, n_channels=8, seed=123)
print(f"Data: {X.shape[0]} trials, {X.shape[1]} channels, {X.shape[2]} samples")

# 训练
pipeline = EndToEndBCIPipeline(sfreq=250.0, n_csp_components=4)
pipeline.fit(X, y)
print("Pipeline trained.")

# 测试
pred, proba = pipeline.predict(X[:5])
for i in range(5):
    label_str = "LEFT MI" if pred[i] == 0 else "RIGHT MI"
    true_str = "LEFT MI" if y[i] == 0 else "RIGHT MI"
    correct = "✓" if pred[i] == y[i] else "✗"
    print(f"  Trial {i+1}: Pred={label_str}, True={true_str}, "
          f"Conf={max(proba[i])*100:.0f}% {correct}")

print("\nPipeline components:")
print(f"  1. Bandpass: 8-30 Hz (4th order Butterworth)")
print(f"  2. CSP:      {pipeline.n_csp_components} spatial filters")
print(f"  3. Features: log-variance of CSP outputs")
print(f"  4. Classifier: LDA (Linear Discriminant Analysis)")

## 单元小结 · 全课程总结

本单元（Unit 6）涵盖：
1. Daisy 16 通道扩展的硬件连接、SDK 命令和通道映射
2. Micro SD 卡离线记录的数据格式和 Python 解析
3. EDF/BDF 标准 EEG 格式的导入导出
4. Cyton 固件架构和 SDK 命令速查表
5. 端到端 BCI 管线整合（预处理 → CSP → LDA → 预测）

### 全课程回顾

```
Unit 1: 硬件架构          → Cyton 组件、ADS1299、10-20 系统
Unit 2: 连接与数据流      → BrainFlow SDK、SDK 命令、数据包解析
Unit 3: 信号处理          → 滤波器、预处理管线、STFT/CWT
Unit 4: EEG 分析          → 频带功率、ERD/ERS、CSP 特征
Unit 5: BCI 实战          → MI-BCI、SSVEP、ITR、实时分类
Unit 6: 高级专题          → Daisy、SD 卡、EDF、固件、端到端管线
```

### 继续探索

- **深度学习 BCI**: EEGNet、DeepConvNet、FBCSP
- **实时系统**: Lab Streaming Layer (LSL)、ROS 集成
- **高级范式**: P300 Speller、c-VEP、Hybrid BCI
- **移动 BCI**: Ganglion + 手机 APP 开发
- **社区**: 访问 [openbci.com/community](https://openbci.com/community) 获取更多灵感

**感谢你的学习！Happy BCI hacking! 🧠⚡**